# user

In [ ]:
VERSION = "0.0.4"
ASSOC = True
PIXEL_SCALE = 0.1

# lib

In [ ]:
from pathlib import Path
import numpy
from pandas import read_csv, concat, interval_range, cut, DataFrame
from tqdm import tqdm
from plots import (
    image_fitting,
    histos,
    residuals,
    case_result_param,
    vignettes,
    case_result,
    DIRS,
    comparaison,
    get_match,
    result_e,
    x_n_seric,
    total_mean_per_shear_bin,
    compute_by_shear_bin,
    ellips_model,
    total_avg,
)


DATASET_DIR_ORIGIN = DIRS["pisco_euclid"] / VERSION
DATASET_DIR = DIRS["pisco_euclid_se"] / VERSION
RESULTS_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR") / f"{VERSION}++"
if ASSOC:
    RESULTS_DIR = RESULTS_DIR.parent / f"{RESULTS_DIR.name}_assoc"
CATALOG_DIR = RESULTS_DIR / "CATALOG"
IMAGES_DIR = RESULTS_DIR / "IMAGES"
PLOTS_DIR = RESULTS_DIR / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# inference

In [ ]:
# load parameters
true_avg = read_csv(DATASET_DIR_ORIGIN / "images_parameters_true.csv", sep=",", skiprows=0)
true = read_csv(DATASET_DIR_ORIGIN / "galaxies_parameters_true.csv", sep=",", skiprows=0)
PRED_FILES = sorted(list((CATALOG_DIR).glob("*_simu.csv")))
TRUE_FILES = sorted(list((DATASET_DIR).glob("*_true_galaxies_params.csv")))

# inference
detections, pred, slice_pred, slice_truth = [[] for _ in range(4)]
pred_e1, pred_e2, pred_e1_err, pred_e2_err = [[] for _ in range(4)]
for i, (res_file, true_file) in enumerate(tqdm(zip(PRED_FILES, TRUE_FILES), total=len(PRED_FILES))):
    # load results
    df_pred = read_csv(res_file).sort_values("x_model")
    df_pred = df_pred.dropna(subset=["e2_model_err", "e1_model_err"])
    detections.append(df_pred.shape[0])
    df_true = read_csv(true_file).sort_values("x")

    # slices
    temp_pred, temp_truth = get_match(df_pred, df_true, radius=2)
    temp_pred["image"] = i
    temp_truth["image"] = i
    slice_pred.append(temp_pred)
    slice_truth.append(temp_truth)

    # predictions galaxies
    df_pred["image"] = i
    pred.append(df_pred)

    # image means
    mu_e1, err_e1 = ellips_model(df_pred["e1_model"], df_pred["e1_model_err"])
    mu_e2, err_e2 = ellips_model(df_pred["e2_model"], df_pred["e2_model_err"])
    pred_e1.append(mu_e1)
    pred_e2.append(mu_e2)
    pred_e1_err.append(err_e1)
    pred_e2_err.append(err_e2)

# results
detections = numpy.array(detections)
pred = concat(pred, ignore_index=True)
pred_avg = DataFrame(
    {
        "e1": numpy.array(pred_e1),
        "e2": numpy.array(pred_e2),
        "e1_err": numpy.array(pred_e1_err),
        "e2_err": numpy.array(pred_e2_err),
    }
)
slice_pred = concat(slice_pred, ignore_index=True)
slice_truth = concat(slice_truth, ignore_index=True)

# post process

In [ ]:
# mask for resnet comparaison
top = 0.05
mask = (true_avg["avg_g1"].abs() <= top) & (true_avg["avg_g2"].abs() <= top)
images_to_keep = mask[mask].index.to_numpy()

# filtering numpy arrays
pred_avg = pred_avg[mask]
true_avg = true_avg[mask]
detections = detections[mask]

# filtering dataframes
slice_pred = slice_pred[slice_pred["image"].isin(images_to_keep)]
slice_truth = slice_truth[slice_truth["image"].isin(images_to_keep)]

pred = pred[pred["image"].isin(images_to_keep)]
true = true[true["image"].isin(images_to_keep)]

# addings columns for shear bins
bin_width_g = 0.01
bin_g = interval_range(start=-0.05, end=0.04 + bin_width_g, freq=bin_width_g, closed="left")
true_avg["avg_g1_bin"] = cut(true_avg["avg_g1"], bins=bin_g)
true_avg["avg_g2_bin"] = cut(true_avg["avg_g2"], bins=bin_g)


print(NB_GAL_TOTAL := true.shape[0])
print(NB_GAL_PRED := pred.shape[0])
print(NB_GOOD_GAL := slice_pred.shape[0])
del df_pred, df_true, mu_e1, mu_e2, err_e1, err_e2, PRED_FILES, TRUE_FILES, i, pred_e1, pred_e2
del res_file, true_file, mask, top, images_to_keep, temp_pred, temp_truth, pred_e1_err, pred_e2_err
# column_pred = ["e1_model", "e2_model", "image"]
# slice_pred_avg_e = slice_pred[column_pred].groupby("image")[column_pred].mean()
# slice_pred_avg_e = slice_pred_avg_e[slice_pred_avg_e["image"].isin(images_to_keep)]
# slice_true_avg_e = slice_true_avg_e[slice_true_avg_e["image"].isin(images_to_keep)]

# histo

In [ ]:
bins = 50
histos(
    detections,
    pred,
    true,
    pred_avg["e1"],
    pred_avg["e2"],
    true_avg,
    PLOTS_DIR,
    PIXEL_SCALE,
    bins,
)
del bins

# results

In [ ]:
case_result(
    pred_avg["e1"],
    pred_avg["e2"],
    true_avg["avg_g1"],
    true_avg["avg_g2"],
    NB_GAL_PRED,
    PLOTS_DIR / "res_SE.pdf",
)

case_result(
    true_avg["avg_obs_e1"],
    true_avg["avg_obs_e2"],
    true_avg["avg_g1"],
    true_avg["avg_g2"],
    NB_GAL_TOTAL,
    PLOTS_DIR / "res_SE_théorique.pdf",
)

total_avg(PLOTS_DIR, pred_avg, true_avg)

# results by images

In [ ]:
CHOICE = 1
# CHOICE = 6578

style_pred = slice_pred.copy()
style_true = slice_truth.copy()

CHOICE_IMAGE = int(numpy.unique(slice_pred["image"])[CHOICE])
example_preds = style_pred[style_pred["image"] == CHOICE_IMAGE].reset_index(drop=True)
example_true = style_true[style_true["image"] == CHOICE_IMAGE].reset_index(drop=True)
comparaison(
    DATASET_DIR / f"{CHOICE_IMAGE:06d}_simu",
    example_preds[
        [
            "fmf_reduced_chi_2",
            "fmf_stop_reason",
            "fmf_flags",
            "x_model",
            "y_model",
            "e1_model",
            "e2_model",
        ]
    ],
    example_true[["obs_e1", "obs_e2", "x", "y"]].sort_values("x"),
    detections[CHOICE],
    true_avg["avg_g1"][CHOICE_IMAGE],
    true_avg["avg_g2"][CHOICE_IMAGE],
    PLOTS_DIR,
)

In [ ]:
if CHOICE <= 100:
    image_fitting(DATASET_DIR, IMAGES_DIR, f"{CHOICE_IMAGE:06d}_simu", PLOTS_DIR)
    image_fitting(DATASET_DIR, IMAGES_DIR, f"{CHOICE_IMAGE:06d}_simu", PLOTS_DIR, transformed=True)
    residuals(DATASET_DIR, IMAGES_DIR, f"{CHOICE_IMAGE:06d}_simu", PLOTS_DIR)
    vignettes(CATALOG_DIR, IMAGES_DIR, f"{CHOICE_IMAGE:06d}_simu", PLOTS_DIR)

# results by params

In [ ]:
bin_width = 0.5
bin_mag = interval_range(start=20, end=24 + bin_width, freq=bin_width, closed="left")
slice_truth["mag_bin"] = cut(slice_truth["mag"], bins=bin_mag)

result_e(PLOTS_DIR, slice_pred, slice_truth, err=0)

In [ ]:
case_result_param(PLOTS_DIR, slice_pred, slice_truth, PIXEL_SCALE)

# results by bin of shear

In [ ]:
result_e1 = compute_by_shear_bin(true_avg, pred, true, true_col="avg_g1", model="e1")
result_e2 = compute_by_shear_bin(true_avg, pred, true, true_col="avg_g2", model="e2")

In [ ]:
total_mean_per_shear_bin(PLOTS_DIR, result_e1, result_e2)

# sersic

In [ ]:
MIN_SERSIC = 0.2
MAX_SERSIC = 6.5
MU_X = -2.6
SIGMA_X = 1.1

x_n_seric(MIN_SERSIC, MAX_SERSIC, MU_X, SIGMA_X, PLOTS_DIR)